In [ ]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


In [3]:
# Molecular Information part

xyz = '''
C    0.751851   -0.378235    0.000000
C    0.000000    0.745928    0.000000
C   -0.751907   -0.378151    0.000000
H    1.694209   -0.880972    0.000000
H    0.000140    1.825086    0.000000
H   -1.694012   -0.881365    0.000000'''
molecule = 'c3h3_cat'
spin = 0
charge =  1


In [4]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 4 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [5]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [ ]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

In [ ]:
n_elec= 2
n_orb = 3 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([9,11,12])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


In [ ]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())